# Task and motion planning on the table top (RO-081)
Every number in the Note comes from this notebook.
1. The world: the cup at A, the bottle left standing on spot B, the goal "cup at B".
2. The task planner alone: its two-action plan and the test that fails.
3. Task and motion planning, sequence before satisfy: skeletons, sampled placements, backtracking (seed 0).
4. The final plan: each skill's configuration and motion.
5. SayCan on the same table: GPT-2 small's scores (Say) times our tests (Can), one skill at a time.

The planner code is `images/tampkit.py` (copied and extended from RO-053's `manip.py`). GPT-2 small runs in NumPy through `tools/gpt2_small.py` (weights cached on the first run, as in DL-087).

In [1]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd

HERE = Path.cwd()
sys.path.insert(0, str(HERE / "images"))
sys.path.insert(0, str(HERE.parents[3] / "tools"))
import tampkit as t

DATA = HERE / "data"
DATA.mkdir(exist_ok=True)
pd.set_option("display.width", 160)

## 1. The world

In [2]:
def span(o, x):
    w = t.SIZE[o][0]
    return (round(x - w / 2, 4), round(x + w / 2, 4))

print("cup at A spans", span("cup", t.A_X), " cup at B would span", span("cup", t.B_X))
print("bottle at B spans", span("bottle", t.B_X))
cupB, botB = span("cup", t.B_X), span("bottle", t.B_X)
overlap = min(cupB[1], botB[1]) - max(cupB[0], botB[0])
print("overlap of the cup at B with the bottle:", round(overlap, 4), "m")
print("bottle cap height above the shoulder:", round(t.TOP + t.SIZE['bottle'][1], 4), "m; cup lid:", round(t.TOP + t.SIZE['cup'][1], 4))
print("reach of the hand: from", round(t.L1 - t.L2, 3), "to", round(t.L1 + t.L2, 3), "m from the shoulder")
assert abs(overlap - 0.04) < 1e-9

cup at A spans (0.15, 0.21)  cup at B would span (0.39, 0.45)
bottle at B spans (0.4, 0.44)
overlap of the cup at B with the bottle: 0.04 m
bottle cap height above the shoulder: 0.04 m; cup lid: -0.02
reach of the hand: from 0.1 to 0.5 m from the shoulder


## 2. The task planner alone (RO-035 forward search, no geometry)

In [3]:
w0 = t.World({"cup": t.A_X, "bottle": t.B_X})
start, goal = {"cup": "A", "bottle": "start"}, {"cup": "B"}
plan1 = next(t.task_plans(start, goal, set()))
print("first skeleton:", plan1)
log1 = []
steps1, fail1 = t.solve_values(w0, start, plan1, np.random.default_rng(0), log=log1)
for e in log1:
    print(e)
print("failure:", fail1)
assert steps1 is None and fail1 == ("cfree", "cup", "B", "bottle", "start")

first skeleton: [('pick', 'cup', 'A'), ('place', 'cup', 'B')]
{'step': 0, 'test': 'motion', 'ok': True, 'o': 'cup', 'x': 0.18, 'length': 154.58748300643694}
{'step': 1, 'o': 'cup', 'x': 0.42, 'sampled': False, 'test': 'cfree', 'ok': False, 'other': 'bottle'}
failure: ('cfree', 'cup', 'B', 'bottle', 'start')


## 3. Task and motion planning: sequence before satisfy, with backtracking (seed 0)

In [4]:
steps, history, log = t.tamp(w0, start, goal, seed=0)
for k, h in enumerate(history, 1):
    print(f"skeleton {k}:", h["plan"], " known collisions:", h["blocked"], " failure:", h["fail"])
L = pd.DataFrame(log)
L["x"] = L["x"].round(4)
L["length"] = L["length"].round(1)
print(L.to_string())
L.to_csv(DATA / "tamp_log.csv", index=False)

skeleton 1: [('pick', 'cup', 'A'), ('place', 'cup', 'B')]  known collisions: set()  failure: ('cfree', 'cup', 'B', 'bottle', 'start')
skeleton 2: [('pick', 'bottle', 'start'), ('place', 'bottle', 'new'), ('pick', 'cup', 'A'), ('place', 'cup', 'B')]  known collisions: {('cup', 'B', 'bottle', 'start')}  failure: None
    step       test     ok       o       x  length sampled   other                           why
0      0     motion   True     cup  0.1800   154.6     NaN     NaN                           NaN
1      1      cfree  False     cup  0.4200     NaN   False  bottle                           NaN
2      0     motion   True  bottle  0.4200    88.7     NaN     NaN                           NaN
3      1     motion   True  bottle  0.3949    11.1    True     NaN                           NaN
4      2     motion   True     cup  0.1800    87.0     NaN     NaN                           NaN
5      3      cfree  False     cup  0.4200     NaN   False  bottle                           NaN
6   

In [5]:
samples = L[L.sampled == True].drop_duplicates(subset="x", keep="first")
print("bottle placements sampled:", len(samples))
for _, r in L[L.sampled == True].iterrows():
    print(round(r.x, 4), r.test, r.ok, r.get("other", ""))
motions = L[L.test == "motion"]
print("motions planned:", len(motions), " of which in the final plan:", len(steps))
# the tests of each rejected sample
x1 = samples.x.iloc[0]
print("sample 1 bottle spans", span("bottle", x1), "; cup at B spans", cupB, "; overlap",
      round(min(cupB[1], span('bottle', x1)[1]) - max(cupB[0], span('bottle', x1)[0]), 4))
x2 = samples.x.iloc[1]
print("sample 2 bottle spans", span("bottle", x2), "; cup at A spans", span("cup", t.A_X), "; overlap",
      round(min(span('cup', t.A_X)[1], span('bottle', x2)[1]) - max(span('cup', t.A_X)[0], span('bottle', x2)[0]), 4))
for x in samples.x.iloc[2:4]:
    cap = t.top("bottle", x)
    print("sample at", x, "cap at", (round(cap[0], 4), cap[1]), "distance from shoulder", round(float(np.hypot(*cap)), 4))
assert len(samples) == 5 and len(history) == 2

bottle placements sampled: 5
0.3949 motion True nan
0.3949 backtrack False nan
0.2076 cfree False cup
0.0909 ik False nan
0.0784 ik False nan
0.4848 motion True nan
motions planned: 7  of which in the final plan: 4
sample 1 bottle spans (np.float64(0.3749), np.float64(0.4149)) ; cup at B spans (0.39, 0.45) ; overlap 0.0249
sample 2 bottle spans (np.float64(0.1876), np.float64(0.2276)) ; cup at A spans (0.15, 0.21) ; overlap 0.0224
sample at 0.0909 cap at (0.0909, 0.04000000000000001) distance from shoulder 0.0993
sample at 0.0784 cap at (0.0784, 0.04000000000000001) distance from shoulder 0.088


## 4. The final plan

In [6]:
rows = []
for s in steps:
    rows.append(dict(skill=f"{s['kind']} {s['o']}", x=round(s['x'], 4), q1=round(s['q'][0], 1), q2=round(s['q'][1], 1),
                     motion_deg=round(s['length'], 1), waypoints=len(s['path'])))
P = pd.DataFrame(rows)
print(P.to_string())
print("total joint motion:", round(sum(s['length'] for s in steps), 1), "degrees")
P.to_csv(DATA / "tamp_plan.csv", index=False)
xb = steps[1]["x"]
print("bottle placed at", round(xb, 4), "spans", span("bottle", xb), "; gap to the cup at B:",
      round(span("bottle", xb)[0] - cupB[1], 4))
# every leg is free in its own world (checked densely)
for s in steps:
    D = np.array(t.densify(s["path"], 0.5))
    assert not s["world"].collide(D[:, 0], D[:, 1]).any()

          skill       x    q1     q2  motion_deg  waypoints
0   pick bottle  0.4200  31.2  -66.4        88.7          2
1  place bottle  0.4848  15.6  -27.3        42.1          2
2      pick cup  0.1800  34.0 -144.1       139.1          5
3     place cup  0.4200  23.2  -67.0       104.6          4
total joint motion: 374.5 degrees
bottle placed at 0.4848 spans (0.4648, 0.5048) ; gap to the cup at B: 0.0148


## 5. SayCan on the table: Say (GPT-2 small) times Can (our tests)
The prompt follows SayCan's format (Ahn et al. 2022, Listing 1): a robot introduction, then Human/Robot examples with numbered steps, with objects other than the cup and the bottle. The language score of a skill is the sum of the log-probabilities GPT-2 gives to its words after the prompt and the steps chosen so far. The affordance is 1 if our tests find a configuration and a motion for the skill (Section 3), 0 if not, and 0.1 for "done" (SayCan's value).

In [7]:
import gpt2_small as g
W = g.load()
PROMPT = """Robot: Hi there, I'm a robot arm working at a table.
Robot: You can ask me to do various tasks and I'll tell you the sequence of actions I would do to accomplish your task.
Human: How would you hold the pen?
Robot: 1. pick up the pen, 2. done.
Human: How would you put the book on the shelf?
Robot: 1. pick up the book, 2. put down the book on the shelf, 3. done.
Human: The box is standing on the mat. How would you put the plate on the mat?
Robot: 1. pick up the box, 2. put down the box on the table, 3. pick up the plate, 4. put down the plate on the mat, 5. done.
Human: How would you move the apple to the tray?
Robot: 1. pick up the apple, 2. put down the apple on the tray, 3. done.
Human: How would you put the cup on spot B?
Robot: """


def lm_score(prefix, skill):
    a, b = g.encode(prefix), g.encode(skill)
    z = g.forward(a + b, W)["logits"].astype(np.float64)
    z = z - z.max(-1, keepdims=True)
    lp = z - np.log(np.exp(z).sum(-1, keepdims=True))
    return float(sum(lp[len(a) - 1 + k, tok] for k, tok in enumerate(b)))


print("GPT-2 small parameters:", f"{g.n_params(W):,}")

GPT-2 small parameters: 124,439,808


In [8]:
rng = np.random.default_rng(0)
world, q, hist, rows, chosen = w0.copy(), t.HOME, "", [], []
for step in range(1, 7):
    prefix = PROMPT + hist + f"{step}. "
    say = {s: lm_score(prefix, s) for s in t.SKILLS}
    tries = {s: t.try_skill(world, q, s, np.random.default_rng(step)) for s in t.SKILLS}
    m = max(say.values())
    share = {s: np.exp(v - m) for s, v in say.items()}
    tot = sum(share.values())
    comb = {s: share[s] / tot * tries[s][0] for s in t.SKILLS}
    ctot = sum(comb.values())
    for s in t.SKILLS:
        rows.append(dict(step=step, skill=s, log_say=round(say[s], 2), say=share[s] / tot, can=tries[s][0],
                         say_x_can=comb[s], say_x_can_share=comb[s] / ctot))
    best = max(comb, key=comb.get)
    lm_only = max(say, key=say.get)
    print(f"step {step}: language alone picks '{lm_only}'; Say x Can picks '{best}'")
    chosen.append(best)
    if best == "done":
        break
    world, q = tries[best][1], tries[best][2]
    hist += f"{step}. {best}, "
S = pd.DataFrame(rows)
S.to_csv(DATA / "saycan_scores.csv", index=False)
print(S.round(4).to_string())
print("SayCan plan:", chosen, " hand holds:", world.held, " cup at B:", world.at.get("cup") == t.B_X)

step 1: language alone picks 'pick up the cup'; Say x Can picks 'pick up the cup'


step 2: language alone picks 'pick up the cup'; Say x Can picks 'done'
    step                             skill  log_say     say  can  say_x_can  say_x_can_share
0      1                   pick up the cup   -12.68  0.9946  1.0     0.9946           0.9995
1      1                pick up the bottle   -22.80  0.0000  1.0     0.0000           0.0000
2      1        put down the cup on spot B   -19.87  0.0007  0.0     0.0000           0.0000
3      1     put down the cup on the table   -26.85  0.0000  0.0     0.0000           0.0000
4      1     put down the bottle on spot B   -29.34  0.0000  0.0     0.0000           0.0000
5      1  put down the bottle on the table   -33.73  0.0000  0.0     0.0000           0.0000
6      1                              done   -18.05  0.0046  0.1     0.0005           0.0005
7      2                   pick up the cup   -12.16  0.9421  0.0     0.0000           0.0000
8      2                pick up the bottle   -21.65  0.0001  0.0     0.0000           0.0000

In [9]:
s2 = S[S.step == 2].set_index("skill")
d, tb = s2.loc["done"], s2.loc["put down the cup on the table"]
print("step 2 log scores: done", d.log_say, " table", tb.log_say)
print("done: share", round(d["say"], 4), "x 0.1 =", round(d["say_x_can"], 5), "; table: share", round(tb["say"], 4), "x 1 =", round(tb["say_x_can"], 5))
print("ratio done / table:", round(d["say_x_can"] / tb["say_x_can"], 3))
print("ratio of language shares table / done:", round(tb["say"] / d["say"], 4))
print("PaLM 540B / GPT-2 small parameters:", round(540e9 / g.n_params(W)))
s1 = S[S.step == 1].set_index("skill")
p1 = np.exp(s1.log_say)
print(f"step 1: p('pick up the cup') = {p1['pick up the cup']:.4g}; the seven add up to {p1.sum():.4g}; share {p1['pick up the cup'] / p1.sum():.4f}")
print(f"step 2: Say share of done {d['say']:.5f}, of table {tb['say']:.6f}; Say x Can {d['say_x_can']:.6f} and {tb['say_x_can']:.6f}; "
      f"language ratio done / table {d['say'] / tb['say']:.2f}")
print(f"step 2: Say share of the two skills Can removed (pick up the cup, put down the cup on spot B): "
      f"{s2.loc['pick up the cup', 'say'] + s2.loc['put down the cup on spot B', 'say']:.4f}; pick up the bottle {s2.loc['pick up the bottle', 'say']:.4f}")

step 2 log scores: done -16.16  table -18.56
done: share 0.0173 x 0.1 = 0.00173 ; table: share 0.0016 x 1 = 0.00157
ratio done / table: 1.104
ratio of language shares table / done: 0.0906
PaLM 540B / GPT-2 small parameters: 4339
step 1: p('pick up the cup') = 3.113e-06; the seven add up to 3.13e-06; share 0.9946
step 2: Say share of done 0.01735, of table 0.001572; Say x Can 0.001735 and 0.001572; language ratio done / table 11.04
step 2: Say share of the two skills Can removed (pick up the cup, put down the cup on spot B): 0.9810; pick up the bottle 0.0001


## Checks

In [10]:
assert chosen[0] == "pick up the cup"
assert history[0]["plan"] == [("pick", "cup", "A"), ("place", "cup", "B")]
assert [a[:2] for a in history[1]["plan"]] == [("pick", "bottle"), ("place", "bottle"), ("pick", "cup"), ("place", "cup")]
print("all checks passed")

all checks passed
